# Procesamiento de texto con LLMs en SQL usando `ai_query()`

Este notebook muestra un patrón clave para la **certificación Databricks Generative AI Engineer Associate**: aplicar un LLM **directamente desde SQL** sobre datos almacenados en **Unity Catalog**, siguiendo una **arquitectura medallion** (Bronze → Gold).

## ¿Qué vas a aprender?

- Ingerir datos en un **Volume** de Unity Catalog.
- Construir la capa **Bronze** (datos crudos) como tabla **Delta**.
- Generar la capa **Gold** (datos enriquecidos) aplicando `ai_query()` para **análisis de sentimiento** y **resúmenes** de reseñas.

## ¿Qué es `ai_query()`?

Es una **función SQL nativa** de Databricks que invoca un *serving endpoint* (un LLM) fila a fila, sin escribir Python. Es la base de los pipelines de IA *batch* en Databricks y un tema central del examen.

```sql
ai_query(endpoint, request [, returnType, failOnError, modelParameters])
```

| Argumento | Descripción |
|-----------|-------------|
| `endpoint` | Nombre del *serving endpoint* (Foundation Model, externo o propio). |
| `request` | Prompt o columna de texto a enviar al modelo. |
| `returnType` | (Opcional) Tipo/esquema de salida, p. ej. `'STRING'` o un `STRUCT` para salida estructurada. |
| `failOnError` | (Opcional) Si `false`, un error en una fila no aborta toda la consulta. |
| `modelParameters` | (Opcional) `struct` con `max_tokens`, `temperature`, etc. |

> 💡 Para tareas comunes existen funciones de IA de más alto nivel que internamente usan `ai_query`: `ai_analyze_sentiment()`, `ai_classify()`, `ai_summarize()`, `ai_translate()`, `ai_extract()`, `ai_gen()`, `ai_similarity()`, `ai_mask()` y `ai_forecast()`.

![AI_Query_Flow](./Assets/ai_query_flow.png)

## 1. Cargar el CSV en un Volume de Unity Catalog

Un **Volume** de Unity Catalog es un espacio de almacenamiento gobernado para archivos no tabulares (CSV, imágenes, PDFs...). Es el lugar recomendado para la ingesta de datos crudos, frente al antiguo DBFS.

Primero creamos el Volume y luego descargamos el CSV de reseñas dentro de él.

In [ ]:
%sql
-- Crea el Volume en el esquema 'default' del catálogo actual (si no existe)
CREATE VOLUME IF NOT EXISTS default.genai_lab;

In [ ]:
import requests

# Obtenemos el catálogo activo para construir las rutas de forma dinámica
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Ruta base del Volume de Unity Catalog
volume_base = f"/Volumes/{catalog_name}/default/genai_lab"

# Descargamos el CSV de reseñas y lo guardamos en el Volume
files = ["restaurant_reviews.csv"]
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/GenAI_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

print(f"Archivo descargado en: {volume_base}")

## 2. Capa Bronze: leer los datos crudos

En la **arquitectura medallion**, la capa **Bronze** contiene los datos tal cual se ingieren, sin transformar. Leemos el CSV desde el Volume en un DataFrame de Spark.

In [ ]:
# Leemos el CSV crudo desde el Volume (header=True usa la primera fila como nombres de columna)
bronze_df = spark.read.load(
    f"/Volumes/{catalog_name}/default/genai_lab/restaurant_reviews.csv",
    format="csv",
    header=True,
)
display(bronze_df.limit(10))

## 3. Guardar la capa Bronze como tabla Delta

Creamos el esquema `bronze` y materializamos el DataFrame como **tabla gestionada Delta** en Unity Catalog. Delta aporta transacciones ACID, *time travel* y rendimiento. Usamos el catálogo activo para que la tabla quede totalmente cualificada (`catálogo.esquema.tabla`).

In [ ]:
# Creamos el esquema 'bronze' en el catálogo activo
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.bronze")

Guardamos como **tabla gestionada** (`saveAsTable`); Unity Catalog administra la ubicación física. `mode("overwrite")` permite re-ejecutar la celda sin errores.

In [ ]:
# Guardamos la capa Bronze como tabla gestionada Delta en Unity Catalog
bronze_table = f"{catalog_name}.bronze.restaurant_reviews"
bronze_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

print(f"Tabla Bronze creada: {bronze_table}")

## 4. Capa Gold: enriquecer los datos con un LLM (`ai_query`)

La capa **Gold** contiene datos listos para negocio. Aquí la generamos aplicando `ai_query()` sobre la columna `Review` para añadir dos columnas nuevas:

- **`sentiment`**: clasificación del sentimiento (Positivo / Neutro / Negativo).
- **`summary`**: resumen de la reseña en una frase.

Cada llamada a `ai_query` envía el texto de la fila al *serving endpoint* y devuelve la respuesta del modelo como una columna más. Bajamos la `temperature` a 0 para que el resultado sea **determinista y consistente** (importante en clasificación).

> 💡 **Concatenación del prompt**: usamos `||` para unir la instrucción con el texto de la columna (`"Clasifica..." || Review`). También se podría usar `modelParameters` para acotar `max_tokens` y reducir coste.

In [ ]:
gold_df = spark.sql(f"""
SELECT
    CustomerID,
    CustomerName,
    Products,
    Review,

    -- Clasificación de sentimiento (respuesta de una sola palabra)
    ai_query(
      "databricks-claude-sonnet-4-5",
      "Clasifica el sentimiento del cliente como Positivo, Neutro o Negativo. Responde solo con una palabra. Reseña: " || Review,
      modelParameters => named_struct('max_tokens', 5, 'temperature', 0.0)
    ) AS sentiment,

    -- Resumen en una frase
    ai_query(
      "databricks-claude-sonnet-4-5",
      "Resume esta reseña de restaurante en una frase corta en español. Reseña: " || Review,
      modelParameters => named_struct('max_tokens', 60, 'temperature', 0.0)
    ) AS summary

FROM {catalog_name}.bronze.restaurant_reviews
""")

display(gold_df.limit(10))

## 5. Guardar la capa Gold como tabla Delta

Creamos el esquema `gold` y persistimos el resultado enriquecido como tabla gestionada Delta, lista para dashboards o análisis.

In [ ]:
# Creamos el esquema 'gold' en el catálogo activo
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.gold")

Persistimos la tabla Gold. Al estar materializada, las llamadas al LLM se ejecutan **una sola vez**; las consultas posteriores leen los resultados ya calculados (sin coste adicional de inferencia).

In [ ]:
# Guardamos la capa Gold como tabla gestionada Delta en Unity Catalog
gold_table = f"{catalog_name}.gold.restaurant_reviews"
gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table)

print(f"Tabla Gold creada: {gold_table}")

## Resumen y puntos clave para la certificación

### Lo que hicimos
1. **Volume** de Unity Catalog → ingesta de archivos crudos.
2. **Bronze** (Delta) → datos sin transformar.
3. **Gold** (Delta) → datos enriquecidos con `ai_query()` (sentimiento + resumen).

### Conceptos que suelen entrar en el examen
- **`ai_query(endpoint, prompt, ...)`** es la forma SQL de invocar un LLM en batch; admite `returnType`, `failOnError` y `modelParameters`.
- **Salida estructurada**: con `returnType` puedes pedir un `STRUCT` y que el modelo devuelva JSON tipado (útil para extracción).
- **Funciones de IA de alto nivel**: `ai_classify`, `ai_summarize`, `ai_analyze_sentiment`, `ai_translate`, `ai_extract`, `ai_gen`, `ai_similarity`, `ai_mask`, `ai_forecast`.
- **Tipos de serving endpoint**: Foundation Model APIs (pay-per-token), Provisioned Throughput (producción) y External Models (proxy a proveedores externos).
- **Gobernanza con Unity Catalog**: catálogos, esquemas, Volumes y tablas Delta gestionadas; nombres totalmente cualificados `catálogo.esquema.tabla`.
- **Coste**: las llamadas al LLM se facturan por token. Materializar la tabla Gold evita reinferir en cada consulta; `max_tokens` acota el gasto.

### Equivalente con función especializada
La clasificación de sentimiento también se podría escribir así, más conciso:
```sql
SELECT Review, ai_analyze_sentiment(Review) AS sentiment
FROM bronze.restaurant_reviews;
```